In [0]:
import pyspark
from pyspark.sql.types import *
import pyspark.sql.functions as f
from pyspark.sql import SparkSession, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import when
from typing import Union, Optional, List
from dataclasses import dataclass
from pyspark.sql.types import IntegerType
from functools import reduce
from datetime import timedelta
from pyspark.sql.functions import broadcast
from pyspark.sql.functions import when, lit

#import
#from pls_common_data_store import pls_data_store
#pds = pls_data_store()

#ignore strange depreciation warnings
from warnings import simplefilter 
simplefilter(action='ignore', category=DeprecationWarning)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
spark.conf.set("spark.sql.shuffle.partitions","auto")
spark.conf.get("spark.sql.shuffle.partitions")

# spark.conf.set("spark.databricks.queryWatchdog.maxQueryTasks", "50000000")

In [0]:
# Processing MCP SSE campaigns + automation
final_result_sse_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_mcp_sse') \
    .dropDuplicates() \
    .filter(f.col("effective_date") >= "20240101") \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("KPM_PROJECT_ID", "kpm_duplicated_id") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_sse_df_test = final_result_sse_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

final_result_sse_df_test.display()

In [0]:
# Processing REM SSE PUSH XCM campaigns + automation
final_result_rem_sse_push_xcm = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_rem_sse_push_xcm') \
    .dropDuplicates() \
    .filter(f.col("effective_date") >= "20240101") \
    .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("KPM_PROJECT_ID", "kpm_duplicated_id") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_rem_sse_push_xcm = final_result_rem_sse_push_xcm.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))
final_result_rem_sse_push_xcm.display()



In [0]:
# Processing XCM and DISP standalone campaigns + automation
final_result_xcm_disp_df_test = spark.read.parquet(f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_xcm_disp') \
    .dropDuplicates() \
    .withColumnRenamed("new_sales_uplift_earned", "abs_sales_uplift") \
    .withColumnRenamed("new_sales_test_earned", "abs_sales_test_earned") \
    .withColumnRenamed("redemption_cost", "new_redemption_cost") \
    .select(
        "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
    )

# Overwrite values for kpm_duplicated_id == 136421 (Hard coding this campaign since it has no barcodes for now, fix later)
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(
    "abs_sales_uplift",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(2813986.5)).otherwise(f.col("abs_sales_uplift"))
).withColumn(
    "adj_iroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(16.1)).otherwise(f.col("adj_iroas"))
).withColumn(
    "abs_iroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(30.4)).otherwise(f.col("abs_iroas"))
).withColumn(
    "adj_aroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(85.0)).otherwise(f.col("adj_aroas"))
).withColumn(
    "abs_aroas",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(161.0)).otherwise(f.col("abs_aroas"))
).withColumn(
    "abs_total_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(92700)).otherwise(f.col("abs_total_cost"))
).withColumn(
    "adj_total_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(175000)).otherwise(f.col("adj_total_cost"))
).withColumn(
    "new_redemption_cost",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(0)).otherwise(f.col("new_redemption_cost"))
).withColumn(
    "abs_sales_test_earned",
    f.when(f.col("kpm_duplicated_id") == 136421, f.lit(14913202)).otherwise(f.col("abs_sales_test_earned"))
)

# Overwrite values for kpm_duplicated_id == 147400 (Hard coding this campaign since it has no barcodes for now, fix later)
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(
    "abs_sales_uplift",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(1718853.8)).otherwise(f.col("abs_sales_uplift"))
).withColumn(
    "adj_iroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(9.0)).otherwise(f.col("adj_iroas"))
).withColumn(
    "abs_iroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(17.5)).otherwise(f.col("abs_iroas"))
).withColumn(
    "adj_aroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(109.0)).otherwise(f.col("adj_aroas"))
).withColumn(
    "abs_aroas",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(210.0)).otherwise(f.col("abs_aroas"))
).withColumn(
    "abs_total_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(97979)).otherwise(f.col("abs_total_cost"))
).withColumn(
    "adj_total_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(190000)).otherwise(f.col("adj_total_cost"))
).withColumn(
    "new_redemption_cost",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(0)).otherwise(f.col("new_redemption_cost"))
).withColumn(
    "abs_sales_test_earned",
    f.when(f.col("kpm_duplicated_id") == 147400, f.lit(20616986)).otherwise(f.col("abs_sales_test_earned"))
)

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

# Remove row where kpm_duplicated_id == 131511 and abs_iroas == 0
final_result_xcm_disp_df_test = final_result_xcm_disp_df_test.filter(~((f.col("kpm_duplicated_id") == 131511) & (f.col("abs_iroas") == 0)))

#136421, 147400
final_result_xcm_disp_df_test.display()

In [0]:
# Processing TDC campaigns + automation (last run: 328 rows)
final_result_tdc_df_test = spark.read.parquet(
  f'abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_tdc'
).dropDuplicates() \
 .withColumnRenamed("abs_sales_uplift_earned", "abs_sales_uplift") \
 .withColumnRenamed("redemption_cost", "new_redemption_cost") \
 .withColumn("adj_sales_total", f.col("adj_sales_total").cast("double")) \
 .select(
    "kpm_duplicated_id", "working_cost", "new_redemption_cost", "adj_sales_uplift", "abs_sales_uplift", "abs_sales_test_earned", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost"
)

# Make negative #s to 0, since we do not report negative #s.
cols_to_zero_neg = [
    "working_cost", "adj_sales_uplift", "abs_sales_uplift", "adj_iroas", "abs_iroas", "adj_aroas", "abs_aroas", "adj_total_cost", "abs_total_cost", "abs_sales_test_earned"
]

for c in cols_to_zero_neg:
    final_result_tdc_df_test = final_result_tdc_df_test.withColumn(c, f.when(f.col(c) < 0, 0).otherwise(f.col(c)))

final_result_tdc_df_test.display()

In [0]:
# Combine TDC, XCM, SSE ABS numbers into one dataframe and write it out
tdc_sse_xcm_disp_abs_df = final_result_tdc_df_test.union(final_result_xcm_disp_df_test).union(
  final_result_sse_df_test).union(final_result_rem_sse_push_xcm)
tdc_sse_xcm_disp_abs_df.display()

tdc_sse_xcm_disp_abs_df.coalesce(1).write.mode("overwrite").parquet('abfss://data-shuttle@sa8451denblrdatamoverdev.dfs.core.windows.net/d199104/kpf_dashboard/absolute_iroas_all_camp_types')